> **NOTE / PREREQUISITE:**
> 1. **Prepare Face Dataset:** Obtain or prepare your target face dataset.
> 2. **Split Large Datasets:** If the dataset is too large to fit in your available storage (Google Drive or Colab environment), split it into smaller batches or subsets using the build_segdeep_dataset.py.
> 3. **Upload to Drive:** Upload the organized dataset (or current batch) to Google Drive before running the pipeline.

In [2]:
import os
import zipfile
from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Configuration paths
ZIP_PATH = "/content/drive/MyDrive/Inpaint/swap5.zip"
DEST_DIR = "/content/muestras"

# Ensure target directory exists
Path(DEST_DIR).mkdir(parents=True, exist_ok=True)

# Extract ZIP file contents
if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DEST_DIR)
    print(f"[INFO] ZIP successfully extracted to: {DEST_DIR}")
else:
    raise FileNotFoundError(f"[ERROR] Specified ZIP file not found at: {ZIP_PATH}")

Mounted at /content/drive
[INFO] ZIP successfully extracted to: /content/muestras


In [ ]:
# ============================================================
# PIXEL-ACCURATE FEATURE MASKS (EYES, NOSE, MOUTH)
# Local Manipulation Deepfake
# ============================================================

# Optional dependency installation cell:
# !pip -q install diffusers transformers accelerate opencv-python pillow tqdm

import random
from pathlib import Path

import cv2
import numpy as np
import torch
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image
from tqdm import tqdm
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

# ============================================================
# PATHS & DIRECTORY SETUP
# ============================================================

REAL_DIR = Path("/content/muestras/swap5")
BASE_OUTPUT = Path("/content/drive/MyDrive/Inpaint/resultados_manipulacion_local")

DIRECTORIES = {
    "images": BASE_OUTPUT / "imagenes_swap",
    "masks": BASE_OUTPUT / "mascaras_swap",
}

for directory in DIRECTORIES.values():
    directory.mkdir(parents=True, exist_ok=True)

# ============================================================
# EXECUTION CONTROLS
# ============================================================

SKIP_COUNT = 8500
SEED = 42
SD_STEPS = 35
GUIDANCE_SCALE = 7.5

# Strength at 0.52 preserves original facial identity
# while modifying internal features (eyes, nose, mouth)
STRENGTH = 0.52

random.seed(SEED)
np.random.seed(SEED)

# Prompt targeted at refining localized features
PROMPT = (
    "highly detailed realistic eyes, detailed iris color, "
    "refined realistic nose shape, detailed natural lips texture, "
    "high resolution photo, 8k"
)

NEGATIVE_PROMPT = (
    "deformed, extra eyes, distorted face, bad anatomy, cartoon, 3d, blurry, ugly"
)

# ============================================================
# 1. LOAD MODELS
# ============================================================

print("\n[1/3] Loading models...")

pipeline = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
).to("cuda")

pipeline.set_progress_bar_config(disable=True)

processor = SegformerImageProcessor.from_pretrained("jonathandinu/face-parsing")
segmenter = SegformerForSemanticSegmentation.from_pretrained(
    "jonathandinu/face-parsing"
).to("cuda")

print("Models loaded successfully.")

# ============================================================
# 2. STRICT FACIAL FEATURE MASK GENERATION
# ============================================================

def get_segmentation(pil_image: Image.Image) -> np.ndarray:
    """Extracts semantic face parsing map logits from input PIL image."""
    inputs = processor(images=pil_image, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = segmenter(**inputs)

    logits = outputs.logits
    logits = torch.nn.functional.interpolate(
        logits, size=pil_image.size[::-1], mode="bilinear", align_corners=False
    )
    return logits.argmax(1)[0].cpu().numpy()


def create_exact_pixel_mask(segmentation: np.ndarray) -> Image.Image | None:
    """
    Extracts pixel-exact regions corresponding to:
    - Left eye (4) & Right eye (5)
    - Nose (10)
    - Mouth/Lips (11, 12, 13)
    Does not apply convex hulls or bounding areas.
    """
    h, w = segmentation.shape
    mask = np.zeros((h, w), dtype=np.uint8)

    # Filter target semantic classes
    mask[
        (segmentation == 4) | (segmentation == 5) |  # Eyes
        (segmentation == 10) |                       # Nose
        (segmentation == 11) | (segmentation == 12) | (segmentation == 13)  # Mouth/Lips
    ] = 255

    if np.sum(mask) == 0:
        return None

    # Minimal 3x3 kernel dilation solely to seam-blend inpainting boundaries
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=1)

    return Image.fromarray(mask)

# ============================================================
# 3. PROCESSING & INPAINTING LOOP
# ============================================================

images = []
for ext in ["*.png", "*.jpg", "*.jpeg"]:
    images.extend(list(REAL_DIR.glob(ext)))
images = sorted(images)

if not images:
    raise RuntimeError(f"No source images found at path: {REAL_DIR}")

random.shuffle(images)

if SKIP_COUNT > 0:
    print(f"Skipping initial {SKIP_COUNT} images out of {len(images)} total...")
    images = images[SKIP_COUNT:]

print(f"\n[2/3] Processing {len(images)} images...")

generator = torch.Generator("cuda").manual_seed(SEED)

print("\n[3/3] Running precision manipulation...")

for img_path in tqdm(images):
    try:
        original_img = Image.open(img_path).convert("RGB")
        orig_w, orig_h = original_img.size

        img_512 = original_img.resize((512, 512), Image.Resampling.LANCZOS)

        # Generate target mask
        seg_map = get_segmentation(img_512)
        feature_mask = create_exact_pixel_mask(seg_map)

        if feature_mask is None:
            print(f"Skipped {img_path.name}: Target features not detected.")
            continue

        # Execute low-strength inpainting pipeline
        inpainted_result = pipeline(
            prompt=PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            image=img_512,
            mask_image=feature_mask,
            strength=STRENGTH,
            num_inference_steps=SD_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
        ).images[0]

        # Resize output back to original dimensions
        inpainted_result = inpainted_result.resize(
            (orig_w, orig_h), Image.Resampling.LANCZOS
        )
        mask_output = feature_mask.resize(
            (orig_w, orig_h), Image.Resampling.LANCZOS
        )

        # Save generated outputs
        inpainted_result.save(DIRECTORIES["images"] / img_path.name)
        mask_output.save(DIRECTORIES["masks"] / img_path.name)

    except Exception as e:
        print(f"Error processing {img_path.name}: {e}")

print(f"\nExecution finished! Outputs saved to:\n{BASE_OUTPUT}")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.



[1/3] Loading models...


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]